In [1]:
from __future__ import annotations

import argparse
import csv
import json
import os
import pickle
import random
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler


@dataclass(frozen=True)
class Paths:
    input_csv: str = "ARQI_2024_complete.csv"
    output_root: str = "outputs"
    prepared_24h: str = "prepared_data_24h_lagged_pm25"
    keras_model: str = "cnn_lstm_keras_24h_lagged_pm25"
    pytorch_model: str = "cnn_lstm_pytorch_48h_lagged_pm25"
    reports: str = "evaluation_reports"


@dataclass(frozen=True)
class DataConfig:
    datetime_col: str = "Datetime"
    target_col: str = "PM2_5_BARC"
    sensor_cols: list[str] = field(default_factory=lambda: [
        "SO2_BARC", "NO_BARC", "NO2_BARC", "NOX_BARC", "CO_BARC", "O3_BARC",
        "PM10_BARC", "WS_BARC", "WD_BARC", "Temp_BARC", "RH_BARC", "BP_BARC", "SR_BARC",
    ])
    time_cols: list[str] = field(default_factory=lambda: [
        "hour_sin", "hour_cos", "month_sin", "month_cos", "dow_sin", "dow_cos",
    ])
    train_ratio: float = 0.70
    val_ratio: float = 0.15
    test_ratio: float = 0.15
    horizon: int = 1

    # Improvement 1: use past PM2.5 values as an input feature.
    # This is not leakage because each window only contains PM2.5 values
    # from the past, while the target is the next hour.
    include_lagged_target: bool = True

    # Improvement 2: give more loss weight to high-pollution target hours
    # so the model cares more about spikes instead of only learning the mean.
    spike_threshold: float = 75.0
    extreme_spike_threshold: float = 150.0
    spike_weight: float = 3.0
    extreme_spike_weight: float = 5.0


@dataclass(frozen=True)
class KerasConfig:
    window_size: int = 24
    cnn_filters_1: int = 64
    cnn_filters_2: int = 32
    cnn_kernel: int = 3
    pool_size: int = 2
    lstm_units_1: int = 100
    lstm_units_2: int = 50
    dense_units: int = 25
    dropout_rate: float = 0.20
    learning_rate: float = 0.001
    batch_size: int = 64
    max_epochs: int = 100
    patience: int = 10
    min_delta: float = 1e-4


@dataclass(frozen=True)
class TorchConfig:
    window_size: int = 48
    cnn_filters_1: int = 64
    cnn_filters_2: int = 32
    cnn_kernel: int = 3
    lstm_units_1: int = 64
    lstm_units_2: int = 32
    dense_units: int = 32
    dropout_rate: float = 0.30
    learning_rate: float = 0.0005
    batch_size: int = 32
    max_epochs: int = 150
    patience: int = 15
    min_delta: float = 1e-4
    warmup_epochs: int = 5
    huber_delta: float = 1.0
    grad_clip: float = 0.5
    seed: int = 42


PATHS = Paths()
DATA = DataConfig()
KERAS = KerasConfig()
TORCH = TorchConfig()


# -----------------------------------------------------------------------------
# File helpers
# -----------------------------------------------------------------------------

def project_dir(*parts: str) -> Path:
    return Path(PATHS.output_root).joinpath(*parts)


def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def save_pickle(obj: Any, path: Path) -> None:
    with path.open("wb") as f:
        pickle.dump(obj, f)


def load_pickle(path: Path) -> Any:
    with path.open("rb") as f:
        return pickle.load(f)


def save_json(obj: dict[str, Any], path: Path) -> None:
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


def print_saved_files(directory: Path) -> None:
    print(f"\nSaved files in: {directory}")
    for path in sorted(directory.iterdir()):
        if path.is_file():
            print(f"  {path.name:<35} {path.stat().st_size / 1024:>9.1f} KB")


# -----------------------------------------------------------------------------
# Data utilities
# -----------------------------------------------------------------------------

def load_arqi_data(input_csv: str = PATHS.input_csv) -> pd.DataFrame:
    df = pd.read_csv(input_csv, parse_dates=[DATA.datetime_col])
    df = df.sort_values(DATA.datetime_col).reset_index(drop=True)
    return df


def validate_no_missing(df: pd.DataFrame, cols: list[str]) -> None:
    missing = int(df[cols].isna().sum().sum())
    if missing != 0:
        raise ValueError(f"Found {missing} missing values. Run imputation first.")


def unique_cols(cols: list[str]) -> list[str]:
    """Preserve order while removing duplicate column names."""
    return list(dict.fromkeys(cols))


def model_feature_cols(add_time_features: bool = False) -> list[str]:
    """
    Feature set used by the forecasting models.

    The important change is that PM2_5_BARC is included as a lagged input.
    Because make_windows() uses rows [t-window+1, ..., t] to predict t+1,
    the model sees only past PM2.5 values, not the target value.
    """
    cols: list[str] = []
    if DATA.include_lagged_target:
        cols.append(DATA.target_col)

    cols.extend(DATA.sensor_cols)

    if add_time_features:
        cols.extend(DATA.time_cols)

    return unique_cols(cols)


def add_cyclic_time_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    dt = out[DATA.datetime_col]

    out["hour_sin"] = np.sin(2 * np.pi * dt.dt.hour / 24)
    out["hour_cos"] = np.cos(2 * np.pi * dt.dt.hour / 24)
    out["month_sin"] = np.sin(2 * np.pi * dt.dt.month / 12)
    out["month_cos"] = np.cos(2 * np.pi * dt.dt.month / 12)
    out["dow_sin"] = np.sin(2 * np.pi * dt.dt.dayofweek / 7)
    out["dow_cos"] = np.cos(2 * np.pi * dt.dt.dayofweek / 7)
    return out


def chronological_split(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    n = len(df)
    train_end = int(n * DATA.train_ratio)
    val_end = int(n * (DATA.train_ratio + DATA.val_ratio))

    train_df = df.iloc[:train_end].copy()
    val_df = df.iloc[train_end:val_end].copy()
    test_df = df.iloc[val_end:].copy()
    return train_df, val_df, test_df


def fit_scalers(
    train_df: pd.DataFrame,
    feature_cols: list[str],
    target_col: str,
) -> tuple[MinMaxScaler, MinMaxScaler]:
    feature_scaler = MinMaxScaler(feature_range=(0, 1))
    target_scaler = MinMaxScaler(feature_range=(0, 1))

    feature_scaler.fit(train_df[feature_cols])
    target_scaler.fit(train_df[[target_col]])
    return feature_scaler, target_scaler


def scale_frame(
    df: pd.DataFrame,
    feature_cols: list[str],
    target_col: str,
    feature_scaler: MinMaxScaler,
    target_scaler: MinMaxScaler,
) -> pd.DataFrame:
    scaled = df.copy()
    scaled[feature_cols] = feature_scaler.transform(df[feature_cols])
    scaled[target_col] = target_scaler.transform(df[[target_col]])
    return scaled


def make_windows(
    df: pd.DataFrame,
    feature_cols: list[str],
    target_col: str,
    window_size: int,
    horizon: int,
) -> tuple[np.ndarray, np.ndarray, list[np.datetime64]]:
    if len(df) < window_size + horizon:
        raise ValueError(
            f"Not enough rows for window_size={window_size} and horizon={horizon}. "
            f"Rows available: {len(df)}."
        )

    features = df[feature_cols].to_numpy(dtype=np.float32)
    target = df[target_col].to_numpy(dtype=np.float32)
    timestamps = df[DATA.datetime_col].to_numpy()

    X, y, ts = [], [], []
    last_start = len(df) - window_size - horizon + 1

    for start in range(last_start):
        target_idx = start + window_size + horizon - 1
        X.append(features[start:start + window_size])
        y.append(target[target_idx])
        ts.append(timestamps[target_idx])

    return np.asarray(X, dtype=np.float32), np.asarray(y, dtype=np.float32), ts


def prepare_windowed_data(
    output_name: str,
    feature_cols: list[str],
    window_size: int,
    add_time_features: bool = False,
) -> Path:
    out_dir = ensure_dir(project_dir(output_name))

    df = load_arqi_data()
    if add_time_features:
        df = add_cyclic_time_features(df)

    required_cols = [DATA.datetime_col, *feature_cols, DATA.target_col]
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise KeyError(f"Missing required columns: {missing_cols}")

    validate_no_missing(df, [*feature_cols, DATA.target_col])
    train_df, val_df, test_df = chronological_split(df)
    feature_scaler, target_scaler = fit_scalers(train_df, feature_cols, DATA.target_col)

    train_sc = scale_frame(train_df, feature_cols, DATA.target_col, feature_scaler, target_scaler)
    val_sc = scale_frame(val_df, feature_cols, DATA.target_col, feature_scaler, target_scaler)
    test_sc = scale_frame(test_df, feature_cols, DATA.target_col, feature_scaler, target_scaler)

    X_train, y_train, ts_train = make_windows(
        train_sc, feature_cols, DATA.target_col, window_size, DATA.horizon
    )
    X_val, y_val, ts_val = make_windows(
        val_sc, feature_cols, DATA.target_col, window_size, DATA.horizon
    )
    X_test, y_test, ts_test = make_windows(
        test_sc, feature_cols, DATA.target_col, window_size, DATA.horizon
    )

    np.save(out_dir / "X_train.npy", X_train)
    np.save(out_dir / "y_train.npy", y_train)
    np.save(out_dir / "X_val.npy", X_val)
    np.save(out_dir / "y_val.npy", y_val)
    np.save(out_dir / "X_test.npy", X_test)
    np.save(out_dir / "y_test.npy", y_test)

    save_pickle(feature_scaler, out_dir / "feature_scaler.pkl")
    save_pickle(target_scaler, out_dir / "target_scaler.pkl")
    save_pickle(ts_train, out_dir / "timestamps_train.pkl")
    save_pickle(ts_val, out_dir / "timestamps_val.pkl")
    save_pickle(ts_test, out_dir / "timestamps_test.pkl")

    config = {
        "input_csv": PATHS.input_csv,
        "datetime_col": DATA.datetime_col,
        "target_col": DATA.target_col,
        "feature_cols": feature_cols,
        "n_features": len(feature_cols),
        "window_size": window_size,
        "horizon": DATA.horizon,
        "train_ratio": DATA.train_ratio,
        "val_ratio": DATA.val_ratio,
        "test_ratio": DATA.test_ratio,
        "train_samples": len(X_train),
        "val_samples": len(X_val),
        "test_samples": len(X_test),
        "date_range": [str(df[DATA.datetime_col].min()), str(df[DATA.datetime_col].max())],
        "uses_cyclic_time_features": add_time_features,
        "uses_lagged_target_feature": DATA.target_col in feature_cols,
        "spike_weighting": {
            "spike_threshold": DATA.spike_threshold,
            "spike_weight": DATA.spike_weight,
            "extreme_spike_threshold": DATA.extreme_spike_threshold,
            "extreme_spike_weight": DATA.extreme_spike_weight,
        },
    }
    save_pickle(config, out_dir / "config.pkl")
    save_json(config, out_dir / "config.json")

    print("\nData prepared")
    print(f"  Output directory : {out_dir}")
    print(f"  Date range       : {config['date_range'][0]} -> {config['date_range'][1]}")
    print(f"  Features         : {len(feature_cols)}")
    print(f"  Window size      : {window_size}")
    print(f"  X_train          : {X_train.shape}")
    print(f"  X_val            : {X_val.shape}")
    print(f"  X_test           : {X_test.shape}")
    print_saved_files(out_dir)
    return out_dir


# -----------------------------------------------------------------------------
# Metrics and reporting
# -----------------------------------------------------------------------------

def compute_metrics(actual: np.ndarray, predicted: np.ndarray) -> dict[str, float]:
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    error = actual - predicted

    mae = float(np.mean(np.abs(error)))
    rmse = float(np.sqrt(np.mean(error ** 2)))

    nonzero = actual != 0
    mape = float(np.mean(np.abs(error[nonzero] / actual[nonzero])) * 100) if nonzero.any() else np.nan

    ss_res = float(np.sum(error ** 2))
    ss_tot = float(np.sum((actual - actual.mean()) ** 2))
    r2 = float(1 - ss_res / ss_tot) if ss_tot > 0 else np.nan

    return {"MAE": mae, "RMSE": rmse, "MAPE": mape, "R2": r2}


def make_spike_sample_weights(
    y_scaled: np.ndarray,
    target_scaler: MinMaxScaler,
    spike_threshold: float = DATA.spike_threshold,
    extreme_spike_threshold: float = DATA.extreme_spike_threshold,
    spike_weight: float = DATA.spike_weight,
    extreme_spike_weight: float = DATA.extreme_spike_weight,
) -> np.ndarray:
    """
    Create sample weights for the target y.

    Normal PM2.5 hours get weight 1.
    PM2.5 >= spike_threshold gets a larger weight.
    PM2.5 >= extreme_spike_threshold gets an even larger weight.

    This helps the model focus more on pollution spikes.
    """
    y_original = target_scaler.inverse_transform(np.asarray(y_scaled).reshape(-1, 1)).ravel()

    weights = np.ones_like(y_original, dtype=np.float32)
    weights[y_original >= spike_threshold] = np.float32(spike_weight)
    weights[y_original >= extreme_spike_threshold] = np.float32(extreme_spike_weight)
    return weights


def print_spike_weight_summary(name: str, y_scaled: np.ndarray, weights: np.ndarray, target_scaler: MinMaxScaler) -> None:
    y_original = target_scaler.inverse_transform(np.asarray(y_scaled).reshape(-1, 1)).ravel()
    n_spike = int((y_original >= DATA.spike_threshold).sum())
    n_extreme = int((y_original >= DATA.extreme_spike_threshold).sum())
    print(
        f"  {name:<10}: samples={len(y_original):>5}, "
        f">= {DATA.spike_threshold:g}={n_spike:>4}, "
        f">= {DATA.extreme_spike_threshold:g}={n_extreme:>4}, "
        f"mean_weight={weights.mean():.3f}"
    )


def save_predictions(
    output_dir: Path,
    timestamps: list[np.datetime64],
    actual: np.ndarray,
    predicted: np.ndarray,
) -> None:
    np.save(output_dir / "predictions_test.npy", predicted)
    np.save(output_dir / "actual_test.npy", actual)
    save_pickle(timestamps, output_dir / "timestamps_test.pkl")

    pred_df = pd.DataFrame({
        "Datetime": pd.to_datetime(timestamps),
        "actual_pm25": actual,
        "predicted_pm25": predicted,
        "error_actual_minus_predicted": actual - predicted,
    })
    pred_df.to_csv(output_dir / "test_predictions.csv", index=False)


def configure_matplotlib() -> None:
    import matplotlib
    matplotlib.use("Agg")


def plot_learning_curves(log_csv: Path, output_path: Path, model_name: str) -> None:
    configure_matplotlib()
    import matplotlib.pyplot as plt

    log = pd.read_csv(log_csv)

    if {"loss", "val_loss", "mae", "val_mae"}.issubset(log.columns):
        x = log["epoch"] + 1 if "epoch" in log.columns else np.arange(1, len(log) + 1)
        train_loss, val_loss = log["loss"], log["val_loss"]
        train_mae, val_mae = log["mae"], log["val_mae"]
    elif {"train_loss", "val_loss", "train_mae", "val_mae"}.issubset(log.columns):
        x = log["epoch"] if "epoch" in log.columns else np.arange(1, len(log) + 1)
        train_loss, val_loss = log["train_loss"], log["val_loss"]
        train_mae, val_mae = log["train_mae"], log["val_mae"]
    else:
        raise ValueError(f"Unsupported training log format: {log_csv}")

    best_epoch = int(x.iloc[int(np.argmin(val_loss))]) if hasattr(x, "iloc") else int(x[np.argmin(val_loss)])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(x, train_loss, label="Train", linewidth=2)
    axes[0].plot(x, val_loss, label="Validation", linewidth=2)
    axes[0].axvline(best_epoch, linestyle="--", alpha=0.7, label=f"Best epoch {best_epoch}")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(x, train_mae, label="Train", linewidth=2)
    axes[1].plot(x, val_mae, label="Validation", linewidth=2)
    axes[1].axvline(best_epoch, linestyle="--", alpha=0.7, label=f"Best epoch {best_epoch}")
    axes[1].set_title("MAE")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("MAE")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.suptitle(f"Training History — {model_name}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close()


def create_evaluation_report(
    model_dir: Path,
    report_name: str,
    model_name: str,
) -> Path:
    configure_matplotlib()
    import matplotlib.dates as mdates
    import matplotlib.pyplot as plt
    from matplotlib.gridspec import GridSpec

    report_dir = ensure_dir(project_dir(PATHS.reports, report_name))

    actual = np.load(model_dir / "actual_test.npy")
    predicted = np.load(model_dir / "predictions_test.npy")

    if (model_dir / "timestamps_test.pkl").exists():
        timestamps = load_pickle(model_dir / "timestamps_test.pkl")
    else:
        timestamps = load_pickle(project_dir(PATHS.prepared_24h) / "timestamps_test.pkl")

    ts = pd.to_datetime(timestamps)
    errors = actual - predicted
    overall = compute_metrics(actual, predicted)

    if (model_dir / "training_log.csv").exists():
        plot_learning_curves(
            model_dir / "training_log.csv",
            report_dir / "learning_curves.png",
            model_name,
        )

    pred_df = pd.DataFrame({
        "Datetime": ts,
        "actual_pm25": actual,
        "predicted_pm25": predicted,
        "error_actual_minus_predicted": errors,
    })
    pred_df.to_csv(report_dir / "test_predictions.csv", index=False)

    fig, ax = plt.subplots(figsize=(16, 5))
    ax.plot(ts, actual, label="Actual PM2.5", linewidth=1.0)
    ax.plot(ts, predicted, label="Predicted PM2.5", linewidth=1.0, linestyle="--")
    ax.axhline(75, linestyle=":", linewidth=1, label="75 µg/m³ threshold")
    ax.set_title(f"Actual vs Predicted PM2.5 — {model_name}")
    ax.set_xlabel("Date")
    ax.set_ylabel("PM2.5 (µg/m³)")
    ax.legend(loc="upper right")
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.savefig(report_dir / "actual_vs_predicted_full.png", dpi=150, bbox_inches="tight")
    plt.close()

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.scatter(actual, predicted, alpha=0.3, s=8, edgecolors="none")
    lo = min(actual.min(), predicted.min()) - 5
    hi = max(actual.max(), predicted.max()) + 5
    ax.plot([lo, hi], [lo, hi], "k--", linewidth=1.5, label="Perfect fit")
    z = np.polyfit(actual, predicted, 1)
    ax.plot(np.sort(actual), np.poly1d(z)(np.sort(actual)), linewidth=1.5, label=f"Fit: y={z[0]:.2f}x+{z[1]:.1f}")
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_xlabel("Actual PM2.5 (µg/m³)")
    ax.set_ylabel("Predicted PM2.5 (µg/m³)")
    ax.set_title(
        f"Scatter — {model_name}\n"
        f"R²={overall['R2']:.4f}  MAE={overall['MAE']:.2f}  RMSE={overall['RMSE']:.2f}"
    )
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(report_dir / "scatter_actual_vs_predicted.png", dpi=150, bbox_inches="tight")
    plt.close()

    fig = plt.figure(figsize=(16, 10))
    gs = GridSpec(2, 3, figure=fig, hspace=0.40, wspace=0.35)

    ax0 = fig.add_subplot(gs[0, :])
    ax0.plot(ts, errors, linewidth=0.7, alpha=0.8)
    ax0.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax0.set_title("Prediction Error Over Time (Actual − Predicted)")
    ax0.set_ylabel("Error (µg/m³)")
    ax0.grid(True, alpha=0.3)
    ax0.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
    plt.setp(ax0.get_xticklabels(), rotation=20)

    ax1 = fig.add_subplot(gs[1, 0])
    ax1.hist(errors, bins=50, edgecolor="white", alpha=0.8, density=True)
    ax1.axvline(0, color="black", linewidth=1, linestyle="--")
    ax1.axvline(errors.mean(), linewidth=1.5, label=f"Mean={errors.mean():.1f}")
    ax1.set_xlabel("Error (µg/m³)")
    ax1.set_ylabel("Density")
    ax1.set_title("Error Distribution")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2 = fig.add_subplot(gs[1, 1])
    ax2.scatter(actual, errors, alpha=0.25, s=6, edgecolors="none")
    ax2.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax2.set_xlabel("Actual PM2.5 (µg/m³)")
    ax2.set_ylabel("Error (µg/m³)")
    ax2.set_title("Error vs Actual Value")
    ax2.grid(True, alpha=0.3)

    hourly = pd.DataFrame({"hour": ts.hour, "error": errors}).groupby("hour")["error"].agg(["mean", "std"])
    ax3 = fig.add_subplot(gs[1, 2])
    ax3.bar(hourly.index, hourly["mean"], alpha=0.8, edgecolor="white")
    ax3.errorbar(hourly.index, hourly["mean"], yerr=hourly["std"], fmt="none", color="black", capsize=3, linewidth=0.8)
    ax3.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax3.set_xlabel("Hour of Day")
    ax3.set_ylabel("Mean Error (µg/m³)")
    ax3.set_title("Diurnal Error Pattern")
    ax3.set_xticks(range(0, 24, 3))
    ax3.grid(True, alpha=0.3, axis="y")

    plt.suptitle(f"Residual Error Analysis — {model_name}", fontsize=14, fontweight="bold")
    plt.savefig(report_dir / "residual_analysis.png", dpi=150, bbox_inches="tight")
    plt.close()

    daily = pred_df.set_index("Datetime")[["actual_pm25", "predicted_pm25"]].resample("D").mean()
    daily_metrics = compute_metrics(daily["actual_pm25"].values, daily["predicted_pm25"].values)

    fig, axes = plt.subplots(2, 1, figsize=(15, 9))
    axes[0].plot(daily.index, daily["actual_pm25"], label="Actual daily mean", linewidth=2, marker="o", markersize=3)
    axes[0].plot(daily.index, daily["predicted_pm25"], label="Predicted daily mean", linewidth=2, linestyle="--", marker="s", markersize=3)
    axes[0].set_title("Daily Mean PM2.5")
    axes[0].set_ylabel("PM2.5 (µg/m³)")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    daily_error = daily["actual_pm25"] - daily["predicted_pm25"]
    axes[1].bar(daily.index, daily_error, alpha=0.8, width=0.8)
    axes[1].axhline(0, color="black", linewidth=0.8, linestyle="--")
    axes[1].set_title("Daily Mean Error")
    axes[1].set_ylabel("Error (µg/m³)")
    axes[1].grid(True, alpha=0.3, axis="y")

    for ax in axes:
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
        ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
        plt.setp(ax.get_xticklabels(), rotation=20)

    plt.suptitle(
        f"Daily Performance — {model_name} | R²={daily_metrics['R2']:.4f} | MAE={daily_metrics['MAE']:.2f} µg/m³",
        fontsize=13,
        fontweight="bold",
    )
    plt.tight_layout()
    plt.savefig(report_dir / "daily_average_performance.png", dpi=150, bbox_inches="tight")
    plt.close()

    month_metrics = {}
    for month in sorted(pred_df["Datetime"].dt.to_period("M").unique()):
        idx = pred_df["Datetime"].dt.to_period("M") == month
        month_metrics[str(month)] = compute_metrics(
            pred_df.loc[idx, "actual_pm25"].values,
            pred_df.loc[idx, "predicted_pm25"].values,
        )

    level_bins = [0, 35, 75, 150, np.inf]
    level_labels = ["Good (<35)", "Moderate (35-75)", "Unhealthy (75-150)", "Very Unhealthy (>150)"]
    pred_df["pm25_level"] = pd.cut(pred_df["actual_pm25"], bins=level_bins, labels=level_labels, right=False)

    level_metrics = {}
    for level in level_labels:
        idx = pred_df["pm25_level"] == level
        if idx.any():
            level_metrics[level] = {
                **compute_metrics(pred_df.loc[idx, "actual_pm25"].values, pred_df.loc[idx, "predicted_pm25"].values),
                "samples": int(idx.sum()),
            }

    all_metrics = {
        "hourly_overall": overall,
        "daily_average": daily_metrics,
        "monthly": month_metrics,
        "by_pm25_level": level_metrics,
    }

    save_pickle(all_metrics, report_dir / "all_metrics.pkl")
    save_json(all_metrics, report_dir / "all_metrics.json")

    print("\nEvaluation complete")
    print(f"  Model       : {model_name}")
    print(f"  Test period : {ts.min()} -> {ts.max()}")
    print(f"  Samples     : {len(actual)}")
    print(f"  MAE         : {overall['MAE']:.4f} µg/m³")
    print(f"  RMSE        : {overall['RMSE']:.4f} µg/m³")
    print(f"  MAPE        : {overall['MAPE']:.4f}%")
    print(f"  R²          : {overall['R2']:.4f}")
    print_saved_files(report_dir)
    return report_dir


# -----------------------------------------------------------------------------
# Keras baseline
# -----------------------------------------------------------------------------

def build_keras_cnn_lstm(window_size: int, n_features: int, cfg: KerasConfig):
    import tensorflow as tf
    from tensorflow.keras.layers import BatchNormalization, Conv1D, Dense, Dropout, Input, LSTM, MaxPooling1D
    from tensorflow.keras.models import Model
    from tensorflow.keras.optimizers import Adam

    inputs = Input(shape=(window_size, n_features), name="input")

    x = Conv1D(cfg.cnn_filters_1, cfg.cnn_kernel, padding="same", activation="relu", name="conv_1")(inputs)
    x = BatchNormalization(name="batch_norm_1")(x)
    x = Conv1D(cfg.cnn_filters_2, cfg.cnn_kernel, padding="same", activation="relu", name="conv_2")(x)
    x = BatchNormalization(name="batch_norm_2")(x)
    x = MaxPooling1D(pool_size=cfg.pool_size, name="max_pool")(x)
    x = Dropout(cfg.dropout_rate, name="dropout_cnn")(x)

    x = LSTM(cfg.lstm_units_1, return_sequences=True, name="lstm_1")(x)
    x = Dropout(cfg.dropout_rate, name="dropout_lstm_1")(x)
    x = LSTM(cfg.lstm_units_2, return_sequences=False, name="lstm_2")(x)
    x = Dropout(cfg.dropout_rate, name="dropout_lstm_2")(x)

    x = Dense(cfg.dense_units, activation="relu", name="dense_1")(x)
    outputs = Dense(1, activation="linear", name="output")(x)

    model = Model(inputs=inputs, outputs=outputs, name="cnn_lstm_keras_pm25")
    model.compile(optimizer=Adam(learning_rate=cfg.learning_rate), loss="mse", metrics=["mae"])
    return model


def train_keras_baseline() -> Path:
    os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

    import tensorflow as tf
    from tensorflow.keras.callbacks import CSVLogger, EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

    prepared_dir = project_dir(PATHS.prepared_24h)
    if not prepared_dir.exists():
        prepare_windowed_data(PATHS.prepared_24h, model_feature_cols(add_time_features=False), KERAS.window_size, add_time_features=False)

    out_dir = ensure_dir(project_dir(PATHS.keras_model))

    X_train = np.load(prepared_dir / "X_train.npy")
    y_train = np.load(prepared_dir / "y_train.npy")
    X_val = np.load(prepared_dir / "X_val.npy")
    y_val = np.load(prepared_dir / "y_val.npy")
    X_test = np.load(prepared_dir / "X_test.npy")
    y_test = np.load(prepared_dir / "y_test.npy")
    ts_test = load_pickle(prepared_dir / "timestamps_test.pkl")
    target_scaler = load_pickle(prepared_dir / "target_scaler.pkl")
    data_config = load_pickle(prepared_dir / "config.pkl")

    train_weights = make_spike_sample_weights(y_train, target_scaler)
    val_weights = make_spike_sample_weights(y_val, target_scaler)

    print("\nSpike-weight summary for Keras training")
    print_spike_weight_summary("train", y_train, train_weights, target_scaler)
    print_spike_weight_summary("val", y_val, val_weights, target_scaler)

    model = build_keras_cnn_lstm(
        window_size=data_config["window_size"],
        n_features=data_config["n_features"],
        cfg=KERAS,
    )

    callbacks = [
        EarlyStopping(
            monitor="val_loss",
            patience=KERAS.patience,
            min_delta=KERAS.min_delta,
            restore_best_weights=True,
            verbose=1,
        ),
        ModelCheckpoint(
            filepath=out_dir / "best_model.keras",
            monitor="val_loss",
            save_best_only=True,
            verbose=1,
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=5,
            min_lr=1e-6,
            verbose=1,
        ),
        CSVLogger(out_dir / "training_log.csv", append=False),
    ]

    print("\nTraining Keras CNN-LSTM 24h with lagged PM2.5 + spike weights")
    print(f"  X_train     : {X_train.shape}")
    print(f"  X_val       : {X_val.shape}")
    print(f"  X_test      : {X_test.shape}")
    print(f"  Output dir  : {out_dir}")
    model.summary()

    history = model.fit(
        X_train,
        y_train,
        sample_weight=train_weights,
        validation_data=(X_val, y_val, val_weights),
        epochs=KERAS.max_epochs,
        batch_size=KERAS.batch_size,
        callbacks=callbacks,
        verbose=1,
    )

    best_epoch = int(np.argmin(history.history["val_loss"]) + 1)
    best_model = tf.keras.models.load_model(out_dir / "best_model.keras")

    predicted_scaled = best_model.predict(X_test, verbose=0)
    actual = target_scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()
    predicted = target_scaler.inverse_transform(predicted_scaled).flatten()
    metrics = compute_metrics(actual, predicted)

    save_predictions(out_dir, ts_test, actual, predicted)

    model_info = {
        "framework": "TensorFlow/Keras",
        "model_name": "cnn_lstm_keras_24h_lagged_pm25_spike_weighted",
        "best_epoch": best_epoch,
        "total_epochs": len(history.history["val_loss"]),
        "best_val_loss": float(min(history.history["val_loss"])),
        "best_val_mae_scaled": float(history.history["val_mae"][best_epoch - 1]),
        "test_metrics": metrics,
        "data_config": data_config,
        "model_config": asdict(KERAS),
        "feature_improvements": [
            "added lagged PM2_5_BARC as an input feature",
            "used sample weights so PM2.5 spikes affect the loss more strongly",
        ],
        "spike_weighting": data_config.get("spike_weighting", {}),
    }
    save_pickle(model_info, out_dir / "metrics.pkl")
    save_json(model_info, out_dir / "metrics.json")

    plot_learning_curves(out_dir / "training_log.csv", out_dir / "training_curves.png", "Keras CNN-LSTM 24h Lagged PM2.5 + Spike Weights")

    print("\nKeras lagged PM2.5 + spike-weighted model complete")
    print(f"  MAE  : {metrics['MAE']:.4f} µg/m³")
    print(f"  RMSE : {metrics['RMSE']:.4f} µg/m³")
    print(f"  MAPE : {metrics['MAPE']:.4f}%")
    print(f"  R²   : {metrics['R2']:.4f}")
    print_saved_files(out_dir)
    return out_dir


# -----------------------------------------------------------------------------
# PyTorch improved model
# -----------------------------------------------------------------------------

def set_reproducibility(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)

    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True
    except Exception:
        pass


def train_pytorch_improved() -> Path:
    import torch
    import torch.nn as nn
    from torch.optim import Adam
    from torch.optim.lr_scheduler import ReduceLROnPlateau
    from torch.utils.data import DataLoader, Dataset

    set_reproducibility(TORCH.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    prepared_name = f"prepared_data_{TORCH.window_size}h_lagged_pm25_time_features"
    prepared_dir = project_dir(prepared_name)
    if not prepared_dir.exists():
        prepare_windowed_data(
            output_name=prepared_name,
            feature_cols=model_feature_cols(add_time_features=True),
            window_size=TORCH.window_size,
            add_time_features=True,
        )

    out_dir = ensure_dir(project_dir(PATHS.pytorch_model))

    X_train = np.load(prepared_dir / "X_train.npy")
    y_train = np.load(prepared_dir / "y_train.npy")
    X_val = np.load(prepared_dir / "X_val.npy")
    y_val = np.load(prepared_dir / "y_val.npy")
    X_test = np.load(prepared_dir / "X_test.npy")
    y_test = np.load(prepared_dir / "y_test.npy")
    ts_test = load_pickle(prepared_dir / "timestamps_test.pkl")
    target_scaler = load_pickle(prepared_dir / "target_scaler.pkl")
    data_config = load_pickle(prepared_dir / "config.pkl")

    train_weights = make_spike_sample_weights(y_train, target_scaler)
    val_weights = make_spike_sample_weights(y_val, target_scaler)
    test_weights = np.ones_like(y_test, dtype=np.float32)

    print("\nSpike-weight summary for PyTorch training")
    print_spike_weight_summary("train", y_train, train_weights, target_scaler)
    print_spike_weight_summary("val", y_val, val_weights, target_scaler)

    class PM25Dataset(Dataset):
        def __init__(self, X: np.ndarray, y: np.ndarray, weights: np.ndarray):
            self.X = torch.tensor(X, dtype=torch.float32)
            self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)
            self.weights = torch.tensor(weights, dtype=torch.float32).unsqueeze(1)

        def __len__(self) -> int:
            return len(self.X)

        def __getitem__(self, idx: int):
            return self.X[idx].permute(1, 0), self.y[idx], self.weights[idx]

    class SimpleCNNLSTM(nn.Module):
        """
        Simpler 48h CNN-LSTM model to reduce overfitting.

        Changes from the previous PyTorch model:
        - no temporal attention layer
        - no bidirectional LSTM
        - smaller LSTM hidden sizes
        - stronger dropout
        - lower learning rate
        - smaller batch size
        """
        def __init__(self, n_features: int, cfg: TorchConfig):
            super().__init__()
            padding = cfg.cnn_kernel // 2

            self.cnn = nn.Sequential(
                nn.Conv1d(n_features, cfg.cnn_filters_1, kernel_size=cfg.cnn_kernel, padding=padding),
                nn.BatchNorm1d(cfg.cnn_filters_1),
                nn.ReLU(),
                nn.Conv1d(cfg.cnn_filters_1, cfg.cnn_filters_2, kernel_size=cfg.cnn_kernel, padding=padding),
                nn.BatchNorm1d(cfg.cnn_filters_2),
                nn.ReLU(),
                nn.MaxPool1d(kernel_size=2),
                nn.Dropout(cfg.dropout_rate),
            )

            # After Conv1D + MaxPool1D, shape is:
            #   (batch, cnn_filters_2, reduced_time)
            # We permute to:
            #   (batch, reduced_time, cnn_filters_2)
            # before feeding into the LSTM.
            self.lstm_1 = nn.LSTM(
                input_size=cfg.cnn_filters_2,
                hidden_size=cfg.lstm_units_1,
                batch_first=True,
                bidirectional=False,
            )
            self.drop_1 = nn.Dropout(cfg.dropout_rate)

            self.lstm_2 = nn.LSTM(
                input_size=cfg.lstm_units_1,
                hidden_size=cfg.lstm_units_2,
                batch_first=True,
                bidirectional=False,
            )
            self.drop_2 = nn.Dropout(cfg.dropout_rate)

            self.fc_1 = nn.Linear(cfg.lstm_units_2, cfg.dense_units)
            self.relu = nn.ReLU()
            self.fc_2 = nn.Linear(cfg.dense_units, 1)

        def forward(self, x):
            x = self.cnn(x)
            x = x.permute(0, 2, 1)
            x, _ = self.lstm_1(x)
            x = self.drop_1(x)
            x, _ = self.lstm_2(x)
            x = x[:, -1, :]
            x = self.drop_2(x)
            x = self.relu(self.fc_1(x))
            return self.fc_2(x)

    train_loader = DataLoader(PM25Dataset(X_train, y_train, train_weights), batch_size=TORCH.batch_size, shuffle=True)
    val_loader = DataLoader(PM25Dataset(X_val, y_val, val_weights), batch_size=TORCH.batch_size, shuffle=False)
    test_loader = DataLoader(PM25Dataset(X_test, y_test, test_weights), batch_size=TORCH.batch_size, shuffle=False)

    model = SimpleCNNLSTM(data_config["n_features"], TORCH).to(device)

    # Weighted MSE makes large PM2.5 spike errors more costly.
    # This is more aggressive for spike learning than Huber loss, which can soften large errors.
    criterion = nn.MSELoss(reduction="none")
    optimizer = Adam(model.parameters(), lr=TORCH.learning_rate / TORCH.warmup_epochs)
    scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=7, min_lr=1e-6)

    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print("\nTraining PyTorch CNN-LSTM 48h with lagged PM2.5 + spike weights")
    print(f"  Device      : {device}")
    print(f"  X_train     : {X_train.shape}")
    print(f"  X_val       : {X_val.shape}")
    print(f"  X_test      : {X_test.shape}")
    print(f"  Parameters  : {total_params:,}")
    print(f"  Output dir  : {out_dir}")

    history = {"train_loss": [], "val_loss": [], "train_mae": [], "val_mae": []}
    best_val_loss = float("inf")
    best_epoch = 0
    patience_count = 0

    log_path = out_dir / "training_log.csv"
    with log_path.open("w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["epoch", "train_loss", "train_mae", "val_loss", "val_mae", "lr"])

        for epoch in range(1, TORCH.max_epochs + 1):
            if epoch <= TORCH.warmup_epochs:
                for group in optimizer.param_groups:
                    group["lr"] = TORCH.learning_rate * epoch / TORCH.warmup_epochs

            model.train()
            train_loss_sum = 0.0
            train_weight_sum = 0.0
            train_mae_sum = 0.0

            for Xb, yb, wb in train_loader:
                Xb = Xb.to(device)
                yb = yb.to(device)
                wb = wb.to(device)

                optimizer.zero_grad()
                preds = model(Xb)
                per_sample_loss = criterion(preds, yb)
                loss = (per_sample_loss * wb).sum() / wb.sum()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), TORCH.grad_clip)
                optimizer.step()

                batch_size = len(Xb)
                train_loss_sum += (per_sample_loss * wb).sum().item()
                train_weight_sum += wb.sum().item()
                train_mae_sum += torch.mean(torch.abs(preds - yb)).item() * batch_size

            train_loss = train_loss_sum / train_weight_sum
            train_mae = train_mae_sum / len(X_train)

            model.eval()
            val_loss_sum = 0.0
            val_weight_sum = 0.0
            val_mae_sum = 0.0

            with torch.no_grad():
                for Xb, yb, wb in val_loader:
                    Xb = Xb.to(device)
                    yb = yb.to(device)
                    wb = wb.to(device)
                    preds = model(Xb)

                    batch_size = len(Xb)
                    per_sample_loss = criterion(preds, yb)
                    val_loss_sum += (per_sample_loss * wb).sum().item()
                    val_weight_sum += wb.sum().item()
                    val_mae_sum += torch.mean(torch.abs(preds - yb)).item() * batch_size

            val_loss = val_loss_sum / val_weight_sum
            val_mae = val_mae_sum / len(X_val)

            if epoch > TORCH.warmup_epochs:
                scheduler.step(val_loss)

            current_lr = optimizer.param_groups[0]["lr"]
            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_loss)
            history["train_mae"].append(train_mae)
            history["val_mae"].append(val_mae)
            writer.writerow([epoch, f"{train_loss:.6f}", f"{train_mae:.6f}", f"{val_loss:.6f}", f"{val_mae:.6f}", f"{current_lr:.8f}"])

            print(
                f"Epoch {epoch:>3}/{TORCH.max_epochs} | "
                f"train={train_loss:.5f} | val={val_loss:.5f} | "
                f"val_mae={val_mae:.5f} | lr={current_lr:.2e}"
            )

            if val_loss < best_val_loss - TORCH.min_delta:
                best_val_loss = val_loss
                best_epoch = epoch
                patience_count = 0
                torch.save(model.state_dict(), out_dir / "best_model.pt")
                print(f"  saved best model at epoch {epoch}")
            else:
                patience_count += 1
                if patience_count >= TORCH.patience and epoch > TORCH.warmup_epochs:
                    print(f"\nEarly stopping at epoch {epoch}")
                    break

    model.load_state_dict(torch.load(out_dir / "best_model.pt", map_location=device))
    model.eval()

    scaled_predictions = []
    scaled_actuals = []
    with torch.no_grad():
        for Xb, yb, _ in test_loader:
            scaled_predictions.append(model(Xb.to(device)).cpu().numpy())
            scaled_actuals.append(yb.numpy())

    predicted_scaled = np.vstack(scaled_predictions)
    actual_scaled = np.vstack(scaled_actuals)
    predicted = target_scaler.inverse_transform(predicted_scaled).flatten()
    actual = target_scaler.inverse_transform(actual_scaled).flatten()
    test_metrics = compute_metrics(actual, predicted)

    save_predictions(out_dir, ts_test, actual, predicted)
    plot_learning_curves(log_path, out_dir / "training_curves.png", "PyTorch CNN-LSTM 48h Lagged PM2.5 + Spike Weights")

    model_info = {
        "framework": "PyTorch",
        "model_name": "cnn_lstm_pytorch_48h_lagged_pm25_spike_weighted",
        "best_epoch": best_epoch,
        "best_val_loss": float(best_val_loss),
        "total_epochs": len(history["val_loss"]),
        "test_metrics": test_metrics,
        "data_config": data_config,
        "model_config": asdict(TORCH),
        "trainable_parameters": total_params,
        "loss_function": "weighted MSE with spike sample weights",
        "feature_improvements": [
            "added lagged PM2_5_BARC as an input feature",
            "used sample weights so PM2.5 spikes affect the loss more strongly",
            "switched PyTorch loss from Huber to weighted MSE to punish spike errors more",
        ],
        "spike_weighting": data_config.get("spike_weighting", {}),
        "regularization_changes": [
            "dropout_rate increased to 0.30",
            "learning_rate reduced to 0.0005",
            "lstm_units_1 reduced to 64",
            "lstm_units_2 reduced to 32",
            "batch_size reduced to 32",
            "removed temporal attention",
            "removed bidirectional LSTM",
        ],
        "kept_from_previous_model": [
            "weighted MSE loss",
            "cyclic time features",
            "48-hour input window",
            "learning-rate warmup",
            "gradient clipping",
        ],
    }
    save_pickle(model_info, out_dir / "metrics.pkl")
    save_json(model_info, out_dir / "metrics.json")
    save_pickle(data_config, out_dir / "config.pkl")
    save_json(data_config, out_dir / "config.json")

    print("\nPyTorch lagged PM2.5 + spike-weighted model complete")
    print(f"  MAE  : {test_metrics['MAE']:.4f} µg/m³")
    print(f"  RMSE : {test_metrics['RMSE']:.4f} µg/m³")
    print(f"  MAPE : {test_metrics['MAPE']:.4f}%")
    print(f"  R²   : {test_metrics['R2']:.4f}")
    print_saved_files(out_dir)
    return out_dir


# -----------------------------------------------------------------------------
# Model comparison
# -----------------------------------------------------------------------------

def compare_models() -> Path:
    configure_matplotlib()
    import matplotlib.dates as mdates
    import matplotlib.pyplot as plt

    keras_dir = project_dir(PATHS.keras_model)
    torch_dir = project_dir(PATHS.pytorch_model)
    out_dir = ensure_dir(project_dir(PATHS.reports, "model_comparison"))

    if not (keras_dir / "test_predictions.csv").exists():
        raise FileNotFoundError("Run train-keras before compare.")
    if not (torch_dir / "test_predictions.csv").exists():
        raise FileNotFoundError("Run train-pytorch before compare.")

    keras_pred = pd.read_csv(keras_dir / "test_predictions.csv", parse_dates=["Datetime"])
    torch_pred = pd.read_csv(torch_dir / "test_predictions.csv", parse_dates=["Datetime"])

    merged = keras_pred.rename(columns={
        "actual_pm25": "actual_keras_window",
        "predicted_pm25": "predicted_keras",
    })[["Datetime", "actual_keras_window", "predicted_keras"]].merge(
        torch_pred.rename(columns={
            "actual_pm25": "actual_pm25",
            "predicted_pm25": "predicted_pytorch",
        })[["Datetime", "actual_pm25", "predicted_pytorch"]],
        on="Datetime",
        how="inner",
    )

    keras_metrics = compute_metrics(merged["actual_pm25"].values, merged["predicted_keras"].values)
    torch_metrics = compute_metrics(merged["actual_pm25"].values, merged["predicted_pytorch"].values)

    comparison = {
        "aligned_samples": int(len(merged)),
        "keras_24h_lagged_pm25_spike_weighted": keras_metrics,
        "pytorch_48h_lagged_pm25_spike_weighted": torch_metrics,
    }
    merged.to_csv(out_dir / "aligned_model_predictions.csv", index=False)
    save_json(comparison, out_dir / "comparison_metrics.json")
    save_pickle(comparison, out_dir / "comparison_metrics.pkl")

    fig, axes = plt.subplots(3, 1, figsize=(16, 13))

    axes[0].plot(merged["Datetime"], merged["actual_pm25"], label="Actual", linewidth=1.0)
    axes[0].plot(merged["Datetime"], merged["predicted_keras"], label="Keras 24h", linewidth=0.9, linestyle="--")
    axes[0].plot(merged["Datetime"], merged["predicted_pytorch"], label="PyTorch 48h lagged PM2.5 + spike weights", linewidth=0.9, linestyle="-.")
    axes[0].set_title("Actual vs Model Predictions")
    axes[0].set_ylabel("PM2.5 (µg/m³)")
    axes[0].legend(loc="upper right")
    axes[0].grid(True, alpha=0.3)
    axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
    plt.setp(axes[0].get_xticklabels(), rotation=20)

    for ax, pred_col, title, metric in [
        (axes[1], "predicted_keras", f"Keras 24h | R²={keras_metrics['R2']:.4f}", keras_metrics),
        (axes[2], "predicted_pytorch", f"PyTorch 48h lagged PM2.5 + spike weights | R²={torch_metrics['R2']:.4f}", torch_metrics),
    ]:
        ax.scatter(merged["actual_pm25"], merged[pred_col], alpha=0.25, s=6, edgecolors="none")
        lo = min(merged["actual_pm25"].min(), merged[pred_col].min()) - 5
        hi = max(merged["actual_pm25"].max(), merged[pred_col].max()) + 5
        ax.plot([lo, hi], [lo, hi], "k--", linewidth=1.3)
        ax.set_title(f"{title} | MAE={metric['MAE']:.2f} | RMSE={metric['RMSE']:.2f}")
        ax.set_xlabel("Actual PM2.5 (µg/m³)")
        ax.set_ylabel("Predicted PM2.5 (µg/m³)")
        ax.set_xlim(lo, hi)
        ax.set_ylim(lo, hi)
        ax.grid(True, alpha=0.3)

    plt.suptitle("CNN-LSTM Model Comparison", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(out_dir / "model_comparison.png", dpi=150, bbox_inches="tight")
    plt.close()

    print("\nModel comparison complete")
    print(f"  Aligned samples : {len(merged)}")
    print(f"  Keras MAE       : {keras_metrics['MAE']:.4f}")
    print(f"  PyTorch MAE     : {torch_metrics['MAE']:.4f}")
    print_saved_files(out_dir)
    return out_dir


# -----------------------------------------------------------------------------
# Command line interface
# -----------------------------------------------------------------------------

def run_command(command: str) -> None:
    if command == "prepare":
        prepare_windowed_data(PATHS.prepared_24h, model_feature_cols(add_time_features=False), KERAS.window_size, add_time_features=False)

    elif command == "train-keras":
        # Train and immediately save the Keras evaluation report.
        train_keras_baseline()
        create_evaluation_report(
            project_dir(PATHS.keras_model),
            "keras_24h_lagged_pm25_spike_weighted",
            "Keras CNN-LSTM 24h Lagged PM2.5 + Spike Weights",
        )

    elif command == "evaluate-keras":
        create_evaluation_report(
            project_dir(PATHS.keras_model),
            "keras_24h_lagged_pm25_spike_weighted",
            "Keras CNN-LSTM 24h Lagged PM2.5 + Spike Weights",
        )

    elif command == "train-pytorch":
        # Train and immediately save the PyTorch evaluation report.
        train_pytorch_improved()
        create_evaluation_report(
            project_dir(PATHS.pytorch_model),
            "pytorch_48h_lagged_pm25_spike_weighted",
            "PyTorch CNN-LSTM 48h Lagged PM2.5 + Spike Weights",
        )

    elif command == "evaluate-pytorch":
        create_evaluation_report(
            project_dir(PATHS.pytorch_model),
            "pytorch_48h_lagged_pm25_spike_weighted",
            "PyTorch CNN-LSTM 48h Lagged PM2.5 + Spike Weights",
        )

    elif command == "compare":
        compare_models()

    elif command == "all":
        prepare_windowed_data(PATHS.prepared_24h, model_feature_cols(add_time_features=False), KERAS.window_size, add_time_features=False)
        train_keras_baseline()
        create_evaluation_report(
            project_dir(PATHS.keras_model),
            "keras_24h_lagged_pm25_spike_weighted",
            "Keras CNN-LSTM 24h Lagged PM2.5 + Spike Weights",
        )
        train_pytorch_improved()
        create_evaluation_report(
            project_dir(PATHS.pytorch_model),
            "pytorch_48h_lagged_pm25_spike_weighted",
            "PyTorch CNN-LSTM 48h Lagged PM2.5 + Spike Weights",
        )
        compare_models()

    else:
        raise ValueError(f"Unknown command: {command}")


# Valid pipeline commands.
VALID_COMMANDS = [
    "prepare",
    "train-keras",
    "evaluate-keras",
    "train-pytorch",
    "evaluate-pytorch",
    "compare",
    "all",
]

# Change this when running inside Colab/Jupyter/PyCharm without terminal arguments.
# Good choices:
#   "prepare"          -> only prepare .npy files
#   "train-keras"      -> train the 24h Keras CNN-LSTM
#   "train-pytorch"    -> train the 48h PyTorch CNN-LSTM
#   "all"              -> run everything and save all reports
DEFAULT_COMMAND = "all"


def running_in_notebook() -> bool:
    """
    Return True in Jupyter/Colab notebooks.

    In notebooks, Python receives extra arguments like:
        -f /root/.local/share/jupyter/runtime/kernel-xxxx.json

    Those arguments are not part of our pipeline, so argparse should not
    read sys.argv there.
    """
    try:
        from IPython import get_ipython

        shell = get_ipython()
        if shell is None:
            return False

        shell_name = shell.__class__.__name__
        return shell_name in {"ZMQInteractiveShell", "Shell"}
    except Exception:
        return False


def parse_args(argv: list[str] | None = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="ARQI PM2.5 CNN-LSTM forecasting pipeline")

    parser.add_argument(
        "command",
        nargs="?",
        default=DEFAULT_COMMAND,
        choices=VALID_COMMANDS,
        help=(
            "Pipeline action to run. "
            f"If omitted in notebook/PyCharm, default is {DEFAULT_COMMAND!r}."
        ),
    )

    # IMPORTANT FIX:
    # In Colab/Jupyter, ignore the notebook kernel arguments.
    if argv is None and running_in_notebook():
        argv = []

    return parser.parse_args(argv)


def main(command: str | None = None) -> None:
    """
    Run the pipeline.

    Examples
    --------
    In a notebook:
        main("prepare")
        main("train-keras")
        main("train-pytorch")   # trains 48h PyTorch and saves its evaluation report
        main("evaluate-pytorch")
        main("compare")

    From terminal:
        python arqi_cnn_lstm_fixed.py prepare
        python arqi_cnn_lstm_fixed.py train-keras
        python arqi_cnn_lstm_fixed.py train-pytorch
    """
    if command is not None:
        if command not in VALID_COMMANDS:
            raise ValueError(f"Unknown command: {command}. Choose from: {VALID_COMMANDS}")
        args = argparse.Namespace(command=command)
    else:
        args = parse_args()

    ensure_dir(Path(PATHS.output_root))
    run_command(args.command)


if __name__ == "__main__":
    main()



Data prepared
  Output directory : outputs/prepared_data_24h_lagged_pm25
  Date range       : 2024-01-01 01:00:00 -> 2025-01-01 00:00:00
  Features         : 14
  Window size      : 24
  X_train          : (6124, 24, 14)
  X_val            : (1294, 24, 14)
  X_test           : (1294, 24, 14)

Saved files in: outputs/prepared_data_24h_lagged_pm25
  X_test.npy                             1698.5 KB
  X_train.npy                            8037.9 KB
  X_val.npy                              1698.5 KB
  config.json                               0.8 KB
  config.pkl                                0.7 KB
  feature_scaler.pkl                        1.2 KB
  target_scaler.pkl                         0.6 KB
  timestamps_test.pkl                      89.8 KB
  timestamps_train.pkl                    424.7 KB
  timestamps_val.pkl                       89.8 KB
  y_test.npy                                5.2 KB
  y_train.npy                              24.0 KB
  y_val.npy                            

Model: "cnn_lstm_keras_pm25"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (None, 24, 14)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_1 (Conv1D)                 │ (None, 24, 64)         │         2,752 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_norm_1                    │ (None, 24, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_2 (Conv1D)                 │ (None, 24, 32)         │         6,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_norm_2                    │ (None, 24, 32)         │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pool (MaxPooling1D)         │ (None, 12, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_cnn (Dropout)           │ (None, 12, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 12, 100)        │        53,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_lstm_1 (Dropout)        │ (None, 12, 100)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 50)             │        30,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_lstm_2 (Dropout)        │ (None, 50)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 25)             │         1,275 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 1)              │            26 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 94,013 (367.24 KB)

 Trainable params: 93,821 (366.49 KB)

 Non-trainable params: 192 (768.00 B)

Epoch 1/100
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - loss: 0.0264 - mae: 0.0744
Epoch 1: val_loss improved from None to 0.00935, saving model to outputs/cnn_lstm_keras_24h_lagged_pm25/best_model.keras

Epoch 1: finished saving model to outputs/cnn_lstm_keras_24h_lagged_pm25/best_model.keras
96/96 ━━━━━━━━━━━━━━━━━━━━ 11s 53ms/step - loss: 0.0185 - mae: 0.0597 - val_loss: 0.0094 - val_mae: 0.0476 - learning_rate: 0.0010
Epoch 2/100
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - loss: 0.0138 - mae: 0.0460
Epoch 2: val_loss did not improve from 0.00935
96/96 ━━━━━━━━━━━━━━━━━━━━ 3s 36ms/step - loss: 0.0114 - mae: 0.0426 - val_loss: 0.0176 - val_mae: 0.1097 - learning_rate: 0.0010
Epoch 3/100
96/96 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - loss: 0.0120 - mae: 0.0425
Epoch 3: val_loss did not improve from 0.00935
96/96 ━━━━━━━━━━━━━━━━━━━━ 3s 36ms/step - loss: 0.0100 - mae: 0.0404 - val_loss: 0.0208 - val_mae: 0.1177 - learning_rate: 0.0010
Epoch 4/100
95/96 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - loss: 

In [8]:
import subprocess, os

print("Current folder:", os.getcwd())

print("\nGit branch:")
subprocess.run(["git", "branch", "--show-current"])

print("\nGit remote:")
subprocess.run(["git", "remote", "-v"])

print("\nGit status:")
subprocess.run(["git", "status"])

print("\nLast commit:")
subprocess.run(["git", "log", "--oneline", "-3"])

Current folder: /content/CNN_LSTM_PM2_5_forecasting_Bangladesh_data

Git branch:

Git remote:

Git status:

Last commit:


CompletedProcess(args=['git', 'log', '--oneline', '-3'], returncode=0)

In [10]:
import os
import subprocess
from pathlib import Path
from getpass import getpass

username = "hungryMatLeon24"
repo = "CNN_LSTM_PM2_5_forecasting_Bangladesh_data"
repo_url = f"https://github.com/{username}/{repo}.git"

# Make sure origin does not contain an old/bad token
subprocess.run(["git", "remote", "set-url", "origin", repo_url], check=True)

token = getpass("Paste your GitHub token here. It will be hidden: ")

askpass_path = Path("/tmp/git_askpass.sh")
askpass_path.write_text(f"""#!/bin/sh
case "$1" in
  *Username*) echo "{username}" ;;
  *Password*) echo "{token}" ;;
esac
""")
askpass_path.chmod(0o700)

env = os.environ.copy()
env["GIT_ASKPASS"] = str(askpass_path)
env["GIT_TERMINAL_PROMPT"] = "0"

branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    text=True
).strip()

print("Pushing branch:", branch)

result = subprocess.run(
    ["git", "push", "origin", branch],
    env=env,
    capture_output=True,
    text=True
)

print("STDOUT:")
print(result.stdout)

print("STDERR:")
print(result.stderr)

if result.returncode == 0:
    print("Push successful.")
else:
    print("Push failed. Copy only the STDERR message here, but do NOT share your token.")

# Remove token from variables
del token

Paste your GitHub token here. It will be hidden: ··········
Pushing branch: main
STDOUT:

STDERR:
To https://github.com/hungryMatLeon24/CNN_LSTM_PM2_5_forecasting_Bangladesh_data.git
 * [new branch]      main -> main

Push successful.
